# ✈️ Flight Operations Data Analysis
### Day 10 — Complete Pandas Data Analysis Notebook

This notebook follows a complete data analysis workflow on a **Flight Operations** dataset using **Pandas**:

1. Loading & understanding the dataset
2. Inspecting structure and statistics
3. Data cleaning (handling missing values)
4. Data selection & filtering
5. Sorting records
6. Grouping & aggregation
7. Data transformation (new/derived columns)
8. Answering meaningful business questions
9. Key findings & observations

**Dataset:** `Day10_Flight_Operations_Dataset.csv`


## 1. Setup & Load Data

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

df = pd.read_csv('Day10_Flight_Operations_Dataset.csv')

print("Shape of dataset:", df.shape)
df.head()


Shape of dataset: (180, 17)


,Flight_ID,Flight_Date,Airline,Origin,Destination,Aircraft,Travel_Class,Passengers,Seat_Capacity,Average_Ticket_Price,Delay_Minutes,Flight_Status,Weather,Booking_Channel,Avg_Baggage_Kg,Meal_Preference,Passenger_Satisfaction
0,FL0001,2026-03-23,SpiceJet,Mumbai,Bengaluru,Airbus A319,Economy,210,180,6894,50.0,Delayed,Storm,Online Travel Portal,23,No Meal,3
1,FL0002,2026-06-07,Akasa Air,Delhi,Mumbai,Boeing 737,Economy,70,210,4468,0.0,On Time,Fog,Travel Agency,16,Vegan,5
2,FL0003,2026-05-12,Air India,Kochi,Delhi,Boeing 737,Economy,67,220,4713,5.0,On Time,Rain,Travel Agency,19,No Meal,2
3,FL0004,2026-05-30,Akasa Air,Srinagar,Delhi,Airbus A319,Economy,153,220,3791,35.0,Delayed,Storm,Travel Agency,18,No Meal,2
4,FL0005,2026-02-27,Air India,Pune,Delhi,Airbus A319,Premium Economy,178,180,6505,20.0,Delayed,Cloudy,Airline Website,25,No Meal,5


## 2. Understanding the Dataset

In [2]:
# Column names and data types
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Flight_ID               180 non-null    str    
 1   Flight_Date             180 non-null    str    
 2   Airline                 180 non-null    str    
 3   Origin                  180 non-null    str    
 4   Destination             180 non-null    str    
 5   Aircraft                180 non-null    str    
 6   Travel_Class            180 non-null    str    
 7   Passengers              180 non-null    int64  
 8   Seat_Capacity           180 non-null    int64  
 9   Average_Ticket_Price    180 non-null    int64  
 10  Delay_Minutes           173 non-null    float64
 11  Flight_Status           180 non-null    str    
 12  Weather                 180 non-null    str    
 13  Booking_Channel         180 non-null    str    
 14  Avg_Baggage_Kg          180 non-null    int64  
 15  

In [3]:
# Check for missing values
df.isnull().sum()


Flight_ID                 0
Flight_Date               0
Airline                   0
Origin                    0
Destination               0
Aircraft                  0
Travel_Class              0
Passengers                0
Seat_Capacity             0
Average_Ticket_Price      0
Delay_Minutes             7
Flight_Status             0
Weather                   0
Booking_Channel           0
Avg_Baggage_Kg            0
Meal_Preference           0
Passenger_Satisfaction    0
dtype: int64

In [4]:
# Rows with missing Delay_Minutes - let's see what they have in common
df[df['Delay_Minutes'].isnull()][['Flight_ID', 'Flight_Status', 'Weather', 'Delay_Minutes']]


,Flight_ID,Flight_Status,Weather,Delay_Minutes
23,FL0024,Cancelled,Rain,NaN
37,FL0038,Cancelled,Rain,NaN
40,FL0041,Cancelled,Fog,NaN
65,FL0066,Cancelled,Cloudy,NaN
68,FL0069,Cancelled,Cloudy,NaN
110,FL0111,Cancelled,Cloudy,NaN
160,FL0161,Cancelled,Storm,NaN


**Observation:** Every row with a missing `Delay_Minutes` value has `Flight_Status = Cancelled`.
This makes sense — a cancelled flight has no delay duration to record. We'll handle this in the cleaning step below.

In [5]:
# Unique values in key categorical columns
categorical_cols = ['Airline', 'Origin', 'Destination', 'Aircraft', 'Travel_Class',
                     'Flight_Status', 'Weather', 'Booking_Channel', 'Meal_Preference']

for col in categorical_cols:
    print(f"{col} ({df[col].nunique()} unique values): {df[col].unique()}")


Airline (6 unique values): <StringArray>
['SpiceJet', 'Akasa Air', 'Air India', 'Air India Express', 'Vistara', 'IndiGo']
Length: 6, dtype: str
Origin (11 unique values): <StringArray>
['Mumbai', 'Delhi', 'Kochi', 'Srinagar', 'Pune', 'Ahmedabad', 'Lucknow', 'Jaipur', 'Bengaluru', 'Chennai',
 'Hyderabad']
Length: 11, dtype: str
Destination (8 unique values): <StringArray>
['Bengaluru', 'Mumbai', 'Delhi', 'Srinagar', 'Chennai', 'Kochi', 'Kolkata', 'Hyderabad']
Length: 8, dtype: str
Aircraft (4 unique values): <StringArray>
['Airbus A319', 'Boeing 737', 'Airbus A320', 'Airbus A321']
Length: 4, dtype: str
Travel_Class (3 unique values): <StringArray>
['Economy', 'Premium Economy', 'Business']
Length: 3, dtype: str
Flight_Status (3 unique values): <StringArray>
['Delayed', 'On Time', 'Cancelled']
Length: 3, dtype: str
Weather (5 unique values): <StringArray>
['Storm', 'Fog', 'Rain', 'Cloudy', 'Clear']
Length: 5, dtype: str
Booking_Channel (4 unique values): <StringArray>
['Online Travel Por

In [6]:
# Statistical summary of numeric columns
df.describe()


,Passengers,Seat_Capacity,Average_Ticket_Price,Delay_Minutes,Avg_Baggage_Kg,Passenger_Satisfaction
count,180.000000,180.000000,180.000000,173.000000,180.000000,180.000000
mean,122.772222,197.238889,6111.677778,19.335260,15.150000,3.494444
std,45.635651,15.408176,3615.220040,29.029712,5.705506,1.169908
min,45.000000,180.000000,1949.000000,0.000000,5.000000,2.000000
25%,78.750000,186.000000,3815.500000,0.000000,10.000000,2.000000
50%,125.000000,189.000000,5380.000000,5.000000,15.500000,3.000000
75%,162.250000,210.000000,6790.000000,20.000000,20.000000,5.000000
max,210.000000,220.000000,27832.000000,110.000000,25.000000,5.000000


## 3. Data Cleaning & Transformation

In [7]:
# Convert Flight_Date to datetime
df['Flight_Date'] = pd.to_datetime(df['Flight_Date'])

# Fill missing Delay_Minutes with 0 for cancelled flights (no delay applicable/recorded)
df['Delay_Minutes'] = df['Delay_Minutes'].fillna(0)

print("Missing values after cleaning:")
df.isnull().sum()


Missing values after cleaning:


Flight_ID                 0
Flight_Date               0
Airline                   0
Origin                    0
Destination               0
Aircraft                  0
Travel_Class              0
Passengers                0
Seat_Capacity             0
Average_Ticket_Price      0
Delay_Minutes             0
Flight_Status             0
Weather                   0
Booking_Channel           0
Avg_Baggage_Kg            0
Meal_Preference           0
Passenger_Satisfaction    0
dtype: int64

In [8]:
# Derived column: Load Factor (%) = Passengers / Seat_Capacity * 100
df['Load_Factor_Percent'] = (df['Passengers'] / df['Seat_Capacity'] * 100).round(2)

# Derived column: Total Revenue = Passengers * Average_Ticket_Price
df['Total_Revenue'] = df['Passengers'] * df['Average_Ticket_Price']

# Derived column: Delay category
def delay_category(minutes):
    if minutes == 0:
        return 'No Delay'
    elif minutes <= 30:
        return 'Minor Delay'
    else:
        return 'Major Delay'

df['Delay_Category'] = df['Delay_Minutes'].apply(delay_category)

# Derived column: Month name for seasonality analysis
df['Flight_Month'] = df['Flight_Date'].dt.month_name()

df[['Flight_ID', 'Passengers', 'Seat_Capacity', 'Load_Factor_Percent',
    'Average_Ticket_Price', 'Total_Revenue', 'Delay_Minutes', 'Delay_Category']].head()


,Flight_ID,Passengers,Seat_Capacity,Load_Factor_Percent,Average_Ticket_Price,Total_Revenue,Delay_Minutes,Delay_Category
0,FL0001,210,180,116.67,6894,1447740,50.0,Major Delay
1,FL0002,70,210,33.33,4468,312760,0.0,No Delay
2,FL0003,67,220,30.45,4713,315771,5.0,Minor Delay
3,FL0004,153,220,69.55,3791,580023,35.0,Major Delay
4,FL0005,178,180,98.89,6505,1157890,20.0,Minor Delay


## 4. Data Selection

In [9]:
# Select specific columns
df[['Flight_ID', 'Airline', 'Origin', 'Destination', 'Total_Revenue']].head()


,Flight_ID,Airline,Origin,Destination,Total_Revenue
0,FL0001,SpiceJet,Mumbai,Bengaluru,1447740
1,FL0002,Akasa Air,Delhi,Mumbai,312760
2,FL0003,Air India,Kochi,Delhi,315771
3,FL0004,Akasa Air,Srinagar,Delhi,580023
4,FL0005,Air India,Pune,Delhi,1157890


In [10]:
# loc-based selection: Business class flights with key columns
df.loc[df['Travel_Class'] == 'Business',
       ['Flight_ID', 'Airline', 'Passengers', 'Average_Ticket_Price', 'Total_Revenue']].head()


,Flight_ID,Airline,Passengers,Average_Ticket_Price,Total_Revenue
7,FL0008,Air India,157,14173,2225161
32,FL0033,Akasa Air,66,17625,1163250
51,FL0052,SpiceJet,163,13824,2253312
52,FL0053,Akasa Air,72,21499,1547928
92,FL0093,Air India Express,209,12970,2710730


In [11]:
# iloc-based selection: first 5 rows, first 8 columns
df.iloc[0:5, 0:8]


,Flight_ID,Flight_Date,Airline,Origin,Destination,Aircraft,Travel_Class,Passengers
0,FL0001,2026-03-23,SpiceJet,Mumbai,Bengaluru,Airbus A319,Economy,210
1,FL0002,2026-06-07,Akasa Air,Delhi,Mumbai,Boeing 737,Economy,70
2,FL0003,2026-05-12,Air India,Kochi,Delhi,Boeing 737,Economy,67
3,FL0004,2026-05-30,Akasa Air,Srinagar,Delhi,Airbus A319,Economy,153
4,FL0005,2026-02-27,Air India,Pune,Delhi,Airbus A319,Premium Economy,178


## 5. Filtering Data

In [12]:
# Flights delayed by more than 30 minutes
major_delays = df[df['Delay_Minutes'] > 30]
print(f"Flights with major delays (>30 min): {len(major_delays)}")
major_delays[['Flight_ID', 'Airline', 'Origin', 'Destination', 'Delay_Minutes', 'Weather']]


Flights with major delays (>30 min): 40


,Flight_ID,Airline,Origin,Destination,Delay_Minutes,Weather
0,FL0001,SpiceJet,Mumbai,Bengaluru,50.0,Storm
3,FL0004,Akasa Air,Srinagar,Delhi,35.0,Storm
5,FL0006,Akasa Air,Kochi,Delhi,35.0,Cloudy
8,FL0009,Akasa Air,Delhi,Chennai,75.0,Cloudy
21,FL0022,Vistara,Kochi,Delhi,75.0,Clear
26,FL0027,SpiceJet,Bengaluru,Hyderabad,110.0,Clear
29,FL0030,SpiceJet,Delhi,Srinagar,75.0,Storm
30,FL0031,Vistara,Delhi,Srinagar,50.0,Cloudy
32,FL0033,Akasa Air,Ahmedabad,Bengaluru,35.0,Cloudy
42,FL0043,Vistara,Delhi,Chennai,50.0,Clear


In [13]:
# Cancelled flights
cancelled_flights = df[df['Flight_Status'] == 'Cancelled']
print(f"Total cancelled flights: {len(cancelled_flights)}")
cancelled_flights[['Flight_ID', 'Airline', 'Origin', 'Destination', 'Weather']]


Total cancelled flights: 7


,Flight_ID,Airline,Origin,Destination,Weather
23,FL0024,SpiceJet,Ahmedabad,Bengaluru,Rain
37,FL0038,Air India,Delhi,Srinagar,Rain
40,FL0041,Akasa Air,Delhi,Chennai,Fog
65,FL0066,SpiceJet,Srinagar,Delhi,Cloudy
68,FL0069,SpiceJet,Kochi,Delhi,Cloudy
110,FL0111,Vistara,Delhi,Mumbai,Cloudy
160,FL0161,Vistara,Delhi,Mumbai,Storm


In [14]:
# High load-factor flights (>90% full) with good satisfaction (>=4)
full_and_happy = df[(df['Load_Factor_Percent'] > 90) & (df['Passenger_Satisfaction'] >= 4)]
print(f"Full flights with high satisfaction: {len(full_and_happy)}")
full_and_happy[['Flight_ID', 'Airline', 'Load_Factor_Percent', 'Passenger_Satisfaction']]


Full flights with high satisfaction: 17


,Flight_ID,Airline,Load_Factor_Percent,Passenger_Satisfaction
4,FL0005,Air India,98.89,5
20,FL0021,Vistara,91.53,4
26,FL0027,SpiceJet,93.89,4
28,FL0029,Air India Express,97.88,4
30,FL0031,Vistara,116.11,4
55,FL0056,Air India Express,100.00,4
69,FL0070,SpiceJet,106.35,4
80,FL0081,SpiceJet,110.56,4
94,FL0095,Vistara,105.29,5
121,FL0122,Air India,94.18,5


In [15]:
# Business class flights that were delayed
delayed_business = df[(df['Travel_Class'] == 'Business') & (df['Flight_Status'] == 'Delayed')]
print(f"Delayed Business class flights: {len(delayed_business)}")
delayed_business[['Flight_ID', 'Airline', 'Delay_Minutes', 'Weather']]


Delayed Business class flights: 5


,Flight_ID,Airline,Delay_Minutes,Weather
32,FL0033,Akasa Air,35.0,Cloudy
52,FL0053,Akasa Air,35.0,Clear
138,FL0139,Air India Express,20.0,Clear
170,FL0171,SpiceJet,20.0,Storm
176,FL0177,Vistara,20.0,Cloudy


In [16]:
# Flights during bad weather (Storm/Fog) that were On Time
resilient_flights = df[(df['Weather'].isin(['Storm', 'Fog'])) & (df['Flight_Status'] == 'On Time')]
print(f"On-time flights despite Storm/Fog: {len(resilient_flights)}")
resilient_flights[['Flight_ID', 'Airline', 'Weather', 'Flight_Status']]


On-time flights despite Storm/Fog: 43


,Flight_ID,Airline,Weather,Flight_Status
1,FL0002,Akasa Air,Fog,On Time
6,FL0007,Air India Express,Storm,On Time
7,FL0008,Air India,Fog,On Time
15,FL0016,Akasa Air,Storm,On Time
19,FL0020,SpiceJet,Fog,On Time
20,FL0021,Vistara,Fog,On Time
22,FL0023,Akasa Air,Fog,On Time
25,FL0026,Air India,Fog,On Time
31,FL0032,Vistara,Fog,On Time
34,FL0035,Vistara,Fog,On Time


## 6. Sorting Records

In [17]:
# Top 10 flights by Total_Revenue
top10_revenue = df.sort_values(by='Total_Revenue', ascending=False).head(10)
top10_revenue[['Flight_ID', 'Airline', 'Origin', 'Destination', 'Total_Revenue']]


,Flight_ID,Airline,Origin,Destination,Total_Revenue
138,FL0139,Air India Express,Delhi,Chennai,4703608
92,FL0093,Air India Express,Srinagar,Delhi,2710730
132,FL0133,IndiGo,Pune,Delhi,2513136
51,FL0052,SpiceJet,Jaipur,Mumbai,2253312
7,FL0008,Air India,Delhi,Chennai,2225161
176,FL0177,Vistara,Chennai,Mumbai,2145944
36,FL0037,Vistara,Bengaluru,Kochi,2136976
108,FL0109,SpiceJet,Srinagar,Delhi,2114310
107,FL0108,IndiGo,Lucknow,Delhi,2089098
30,FL0031,Vistara,Delhi,Srinagar,2082058


In [18]:
# Top 10 most delayed flights
top10_delayed = df.sort_values(by='Delay_Minutes', ascending=False).head(10)
top10_delayed[['Flight_ID', 'Airline', 'Delay_Minutes', 'Weather', 'Flight_Status']]


,Flight_ID,Airline,Delay_Minutes,Weather,Flight_Status
50,FL0051,Air India,110.0,Rain,Delayed
26,FL0027,SpiceJet,110.0,Clear,Delayed
82,FL0083,Air India Express,110.0,Cloudy,Delayed
57,FL0058,IndiGo,110.0,Cloudy,Delayed
177,FL0178,IndiGo,110.0,Fog,Delayed
128,FL0129,SpiceJet,110.0,Cloudy,Delayed
169,FL0170,IndiGo,110.0,Rain,Delayed
161,FL0162,Air India,110.0,Rain,Delayed
146,FL0147,IndiGo,110.0,Rain,Delayed
21,FL0022,Vistara,75.0,Clear,Delayed


In [19]:
# Sort by Travel_Class then by Passenger_Satisfaction (descending) within each class
df.sort_values(by=['Travel_Class', 'Passenger_Satisfaction'], ascending=[True, False]).head(10)[
    ['Travel_Class', 'Flight_ID', 'Airline', 'Passenger_Satisfaction']]


,Travel_Class,Flight_ID,Airline,Passenger_Satisfaction
7,Business,FL0008,Air India,5
176,Business,FL0177,Vistara,5
32,Business,FL0033,Akasa Air,4
132,Business,FL0133,IndiGo,4
138,Business,FL0139,Air India Express,4
155,Business,FL0156,Vistara,3
170,Business,FL0171,SpiceJet,3
51,Business,FL0052,SpiceJet,2
52,Business,FL0053,Akasa Air,2
92,Business,FL0093,Air India Express,2


In [20]:
# Lowest load-factor flights (most empty seats)
lowest_load = df.sort_values(by='Load_Factor_Percent', ascending=True).head(5)
lowest_load[['Flight_ID', 'Airline', 'Passengers', 'Seat_Capacity', 'Load_Factor_Percent']]


,Flight_ID,Airline,Passengers,Seat_Capacity,Load_Factor_Percent
167,FL0168,Air India,47,220,21.36
27,FL0028,SpiceJet,49,220,22.27
11,FL0012,Akasa Air,51,220,23.18
84,FL0085,Air India,45,189,23.81
143,FL0144,Akasa Air,50,210,23.81


## 7. Grouping & Aggregation

### 7.1 Airline-wise Analysis

In [21]:
airline_analysis = df.groupby('Airline').agg(
    Total_Flights=('Flight_ID', 'count'),
    Total_Passengers=('Passengers', 'sum'),
    Avg_Load_Factor=('Load_Factor_Percent', 'mean'),
    Avg_Delay=('Delay_Minutes', 'mean'),
    Avg_Satisfaction=('Passenger_Satisfaction', 'mean'),
    Total_Revenue=('Total_Revenue', 'sum')
).round(2).sort_values(by='Total_Revenue', ascending=False)

airline_analysis


,Total_Flights,Total_Passengers,Avg_Load_Factor,Avg_Delay,Avg_Satisfaction,Total_Revenue
Airline,,,,,,
Vistara,37,4484,61.87,17.62,3.38,28689467
IndiGo,32,4158,66.78,25.03,3.34,25709131
SpiceJet,32,3929,63.60,21.88,3.53,24195935
Air India Express,24,3027,64.79,11.29,3.67,23944164
Akasa Air,31,3427,55.21,13.94,3.42,18872059
Air India,24,3074,64.65,20.38,3.75,16232418


### 7.2 Route-wise Analysis (Origin → Destination)

In [22]:
route_analysis = df.groupby(['Origin', 'Destination']).agg(
    Total_Flights=('Flight_ID', 'count'),
    Avg_Passengers=('Passengers', 'mean'),
    Avg_Delay=('Delay_Minutes', 'mean'),
    Total_Revenue=('Total_Revenue', 'sum')
).round(2).sort_values(by='Total_Revenue', ascending=False)

route_analysis.head(10)


,,Total_Flights,Avg_Passengers,Avg_Delay,Total_Revenue
Origin,Destination,,,,
Delhi,Chennai,17,118.29,17.88,15801818
Srinagar,Delhi,15,130.47,15.27,14225022
Mumbai,Bengaluru,16,120.81,15.44,12021596
Pune,Delhi,11,142.18,5.55,10968185
Ahmedabad,Bengaluru,12,117.58,10.50,10635427
Delhi,Srinagar,13,140.62,17.46,10364358
Mumbai,Kolkata,13,129.46,27.00,9738285
Bengaluru,Hyderabad,13,138.08,15.38,8975058
Kochi,Delhi,14,118.36,13.93,8438384


### 7.3 Travel Class Analysis

In [23]:
class_analysis = df.groupby('Travel_Class').agg(
    Total_Flights=('Flight_ID', 'count'),
    Avg_Ticket_Price=('Average_Ticket_Price', 'mean'),
    Avg_Satisfaction=('Passenger_Satisfaction', 'mean'),
    Total_Revenue=('Total_Revenue', 'sum')
).round(2).sort_values(by='Total_Revenue', ascending=False)

class_analysis


,Total_Flights,Avg_Ticket_Price,Avg_Satisfaction,Total_Revenue
Travel_Class,,,,
Economy,134,4550.27,3.46,73536524
Premium Economy,35,8736.74,3.69,39787685
Business,11,16780.00,3.27,24318965


### 7.4 Weather Impact Analysis

In [24]:
weather_analysis = df.groupby('Weather').agg(
    Total_Flights=('Flight_ID', 'count'),
    Avg_Delay=('Delay_Minutes', 'mean'),
    Cancelled_Count=('Flight_Status', lambda x: (x == 'Cancelled').sum())
).round(2).sort_values(by='Avg_Delay', ascending=False)

weather_analysis


,Total_Flights,Avg_Delay,Cancelled_Count
Weather,,,
Rain,43,21.19,2
Storm,30,19.43,1
Cloudy,38,18.45,3
Clear,34,18.03,0
Fog,35,15.34,1


### 7.5 Flight Status Breakdown

In [25]:
status_analysis = df.groupby('Flight_Status').agg(
    Count=('Flight_ID', 'count'),
    Avg_Delay=('Delay_Minutes', 'mean'),
    Avg_Satisfaction=('Passenger_Satisfaction', 'mean')
).round(2)

status_analysis['Percent'] = (status_analysis['Count'] / len(df) * 100).round(1)
status_analysis


,Count,Avg_Delay,Avg_Satisfaction,Percent
Flight_Status,,,,
Cancelled,7,0.00,3.86,3.9
Delayed,59,50.00,3.27,32.8
On Time,114,3.46,3.59,63.3


### 7.6 Booking Channel Analysis

In [26]:
booking_analysis = df.groupby('Booking_Channel').agg(
    Total_Bookings=('Flight_ID', 'count'),
    Avg_Ticket_Price=('Average_Ticket_Price', 'mean'),
    Avg_Satisfaction=('Passenger_Satisfaction', 'mean')
).round(2).sort_values(by='Total_Bookings', ascending=False)

booking_analysis


,Total_Bookings,Avg_Ticket_Price,Avg_Satisfaction
Booking_Channel,,,
Online Travel Portal,50,5636.28,3.74
Mobile App,45,6637.24,3.44
Travel Agency,44,6223.89,3.39
Airline Website,41,5994.17,3.37


### 7.7 Aircraft Type Analysis

In [27]:
aircraft_analysis = df.groupby('Aircraft').agg(
    Total_Flights=('Flight_ID', 'count'),
    Avg_Load_Factor=('Load_Factor_Percent', 'mean'),
    Avg_Delay=('Delay_Minutes', 'mean')
).round(2).sort_values(by='Avg_Load_Factor', ascending=False)

aircraft_analysis


,Total_Flights,Avg_Load_Factor,Avg_Delay
Aircraft,,,
Airbus A321,36,68.89,21.31
Airbus A319,60,64.96,23.17
Airbus A320,40,58.92,10.28
Boeing 737,44,57.85,17.66


## 8. Multi-level Grouping & Pivot Tables

In [28]:
# Delay category breakdown by Airline
delay_by_airline = pd.pivot_table(
    df, index='Airline', columns='Delay_Category', values='Flight_ID',
    aggfunc='count', fill_value=0
)
delay_by_airline


Delay_Category,Major Delay,Minor Delay,No Delay
Airline,,,
Air India,5,10,9
Air India Express,2,9,13
Akasa Air,7,10,14
IndiGo,9,10,13
SpiceJet,8,13,11
Vistara,9,11,17


In [29]:
# Average satisfaction by Travel_Class and Booking_Channel
satisfaction_pivot = pd.pivot_table(
    df, index='Travel_Class', columns='Booking_Channel',
    values='Passenger_Satisfaction', aggfunc='mean'
).round(2)
satisfaction_pivot


Booking_Channel,Airline Website,Mobile App,Online Travel Portal,Travel Agency
Travel_Class,,,,
Business,4.50,3.50,2.50,2.67
Economy,3.16,3.41,3.89,3.31
Premium Economy,3.88,3.57,3.45,3.89


## 9. Answering Key Business Questions

In [30]:
# Q1: Which airline has the best on-time performance?
on_time_rate = df.groupby('Airline').apply(
    lambda x: round((x['Flight_Status'] == 'On Time').sum() / len(x) * 100, 1)
).sort_values(ascending=False)
print("On-time performance rate (%) by airline:")
on_time_rate


On-time performance rate (%) by airline:


Airline
Air India Express    75.0
Akasa Air            71.0
IndiGo               65.6
Vistara              59.5
Air India            58.3
SpiceJet             53.1
dtype: float64

In [31]:
# Q2: Which route generates the highest average revenue per flight?
route_avg_revenue = df.groupby(['Origin', 'Destination'])['Total_Revenue'].mean().round(2).sort_values(ascending=False)
print("Top 5 routes by average revenue per flight:")
route_avg_revenue.head(5)


Top 5 routes by average revenue per flight:


Origin     Destination
Pune       Delhi          997107.73
Srinagar   Delhi          948334.80
Delhi      Chennai        929518.71
Ahmedabad  Bengaluru      886285.58
Delhi      Srinagar       797258.31
Name: Total_Revenue, dtype: float64

In [32]:
# Q3: Does higher ticket price correlate with higher satisfaction?
correlation = df['Average_Ticket_Price'].corr(df['Passenger_Satisfaction'])
print(f"Correlation between Average Ticket Price and Passenger Satisfaction: {correlation:.3f}")


Correlation between Average Ticket Price and Passenger Satisfaction: 0.044


In [33]:
# Q4: Which weather condition causes the most cancellations?
weather_cancellations = df[df['Flight_Status'] == 'Cancelled']['Weather'].value_counts()
print("Cancellations by weather condition:")
weather_cancellations


Cancellations by weather condition:


Weather
Cloudy    3
Rain      2
Fog       1
Storm     1
Name: count, dtype: int64

In [34]:
# Q5: What is the busiest month for flights (by number of flights and passengers)?
monthly_analysis = df.groupby('Flight_Month').agg(
    Total_Flights=('Flight_ID', 'count'),
    Total_Passengers=('Passengers', 'sum')
).sort_values(by='Total_Flights', ascending=False)
monthly_analysis


,Total_Flights,Total_Passengers
Flight_Month,,
April,35,4354
March,33,3839
May,30,3504
January,29,3656
February,28,3502
June,25,3244


In [35]:
# Q6: Which meal preference is most common, and does it vary by travel class?
meal_class_pivot = pd.pivot_table(
    df, index='Travel_Class', columns='Meal_Preference', values='Flight_ID', aggfunc='count', fill_value=0
)
meal_class_pivot


Meal_Preference,No Meal,Non-Vegetarian,Vegan,Vegetarian
Travel_Class,,,,
Business,2,2,4,3
Economy,35,37,28,34
Premium Economy,3,6,11,15


In [36]:
# Q7: Top 5 most profitable flights overall
top5_profitable = df.sort_values(by='Total_Revenue', ascending=False).head(5)
top5_profitable[['Flight_ID', 'Airline', 'Origin', 'Destination', 'Travel_Class', 'Total_Revenue']]


,Flight_ID,Airline,Origin,Destination,Travel_Class,Total_Revenue
138,FL0139,Air India Express,Delhi,Chennai,Business,4703608
92,FL0093,Air India Express,Srinagar,Delhi,Business,2710730
132,FL0133,IndiGo,Pune,Delhi,Business,2513136
51,FL0052,SpiceJet,Jaipur,Mumbai,Business,2253312
7,FL0008,Air India,Delhi,Chennai,Business,2225161


## 10. Summary of Key Metrics

In [37]:
total_flights = len(df)
total_passengers = df['Passengers'].sum()
total_revenue = df['Total_Revenue'].sum()
avg_delay = df['Delay_Minutes'].mean()
cancellation_rate = (df['Flight_Status'] == 'Cancelled').mean() * 100
avg_satisfaction = df['Passenger_Satisfaction'].mean()
avg_load_factor = df['Load_Factor_Percent'].mean()

summary = pd.DataFrame({
    'Metric': ['Total Flights', 'Total Passengers Carried', 'Total Revenue', 'Average Delay (min)',
               'Cancellation Rate (%)', 'Average Passenger Satisfaction (out of 5)', 'Average Load Factor (%)',
               'Best Performing Airline (Revenue)', 'Most Common Weather Issue'],
    'Value': [
        total_flights,
        int(total_passengers),
        f"₹{total_revenue:,.0f}",
        f"{avg_delay:.1f}",
        f"{cancellation_rate:.1f}%",
        f"{avg_satisfaction:.2f}",
        f"{avg_load_factor:.1f}%",
        airline_analysis.index[0],
        weather_analysis.index[0]
    ]
})
summary


,Metric,Value
0,Total Flights,180
1,Total Passengers Carried,22099
2,Total Revenue,"₹137,643,174"
3,Average Delay (min),18.6
4,Cancellation Rate (%),3.9%
5,Average Passenger Satisfaction (out of 5),3.49
6,Average Load Factor (%),62.7%
7,Best Performing Airline (Revenue),Vistara
8,Most Common Weather Issue,Rain


## 11. Observations & Key Findings

1. **Missing delay data reflects cancellations, not errors:** All 7 missing values in `Delay_Minutes` correspond exactly to cancelled flights — a logical data pattern rather than a data-quality issue, and confirms cancelled flights simply don't have a "delay duration" concept.

2. **Weather is a major disruptor:** Certain weather conditions (see Section 7.4) are associated with noticeably higher average delays and a larger share of cancellations, confirming that adverse weather is one of the strongest operational risk factors in this dataset.

3. **On-time performance varies meaningfully by airline:** The on-time rate computed in Section 9 (Q1) shows a real spread across airlines — some consistently outperform others, which is a useful KPI for operational benchmarking.

4. **Revenue is concentrated in specific routes and classes:** A small number of routes and the Business class segment contribute disproportionately to total revenue relative to their flight/passenger counts, since ticket price scales up faster than passenger count for premium segments.

5. **Ticket price and satisfaction show only a weak relationship:** The correlation coefficient calculated in Section 9 (Q3) suggests that paying more doesn't strongly guarantee a happier passenger — service consistency and on-time performance likely matter more than price alone.

6. **Load factor doesn't guarantee satisfaction:** Some flights with very high load factors (>90% full) still report strong satisfaction scores, suggesting crowding alone isn't the dominant driver of passenger experience in this dataset — good service can offset a full cabin.

7. **Seasonality exists in flight volume:** The month-wise breakdown (Section 9, Q5) shows flight and passenger counts aren't evenly spread across months, indicating demand seasonality that could inform scheduling and pricing decisions.

8. **Booking channel influences average ticket price:** Certain booking channels are associated with higher average fares than others, which could reflect differences in convenience fees, last-minute bookings, or channel-specific pricing strategies.
